# 날짜 가이드 박스 1차 실험: 원본 전체 vs 중앙 자르기

"직원이 날짜를 화면 가운데 가이드 박스에 맞춰 찍는다"를 **기존 사진을 자르는 방식**으로 흉내 낸다.

- **사진:** 화장품 개발용 900001~900300 중, 날짜 글자가 가운데 **40% 영역에도 여유 있게 들어온 142장**에서 **무작위 50장**(시드 20260929)
  - 사진 위치와 라벨로만 골랐고, OCR 결과는 보지 않았다(`center_selection_900001-900300.csv`)
  - 40% 안이면 60% 안에도 들어오므로, 50장 모두 세 가지를 같은 사진으로 비교한다
- **비교:** 50장 × (원본 전체, **가운데 60%**, **가운데 40%**)
- **설정:** 지금 제출 파이프라인 그대로(OCR·Parser·재시도 같음). 사진별로 날짜에 맞춰 수동으로 자르지 않는다.
- **순서:** 사진마다 원본/자른 것의 실행 순서를 무작위로 섞는다(노트북 속도 변화가 한쪽에만 쏠리지 않게).
- **예상 시간:** 약 1~1.5시간. 중간에 멈추면 4번 셀만 다시 실행한다(끝난 것은 건너뜀).
- **전원 연결**, 노트북 뚜껑 열어 두기. 주피터는 한 개만.

**끝나면** 5번 셀 결과 화면과 `center_experiment\center_ocr.jsonl`을 Claude에게 보낸다.

In [ ]:
# 1) 설정
from pathlib import Path
WORK_DIR  = r"C:\Users\user\itda_OCR"
COS_DIR   = WORK_DIR + r"\Cosmetics"
COS_LABEL = WORK_DIR + r"\cosmetics_labels_900001-900300.csv"
OUT_DIR   = WORK_DIR + r"\center_experiment"
MODE      = "full-stage"        # 제출 경로 시간은 따로 재고, 나머지 단계도 저장해 둔다(분석용)
IDS   = ["900003", "900004", "900020", "900035", "900036", "900037", "900047", "900049", "900060", "900078", "900084", "900097", "900123", "900126", "900129", "900130", "900133", "900136", "900138", "900141", "900151", "900163", "900173", "900175", "900183", "900185", "900191", "900193", "900198", "900202", "900205", "900210", "900220", "900229", "900238", "900241", "900251", "900254", "900255", "900260", "900265", "900267", "900269", "900271", "900272", "900277", "900281", "900291", "900296", "900297"]
IDS40 = ["900003", "900004", "900020", "900035", "900036", "900037", "900047", "900049", "900060", "900078", "900084", "900097", "900123", "900126", "900129", "900130", "900133", "900136", "900138", "900141", "900151", "900163", "900173", "900175", "900183", "900185", "900191", "900193", "900198", "900202", "900205", "900210", "900220", "900229", "900238", "900241", "900251", "900254", "900255", "900260", "900265", "900267", "900269", "900271", "900272", "900277", "900281", "900291", "900296", "900297"]
VARIANTS = {"full": None, "center60": 0.60, "center40": 0.40}
SEED = 20260929

In [ ]:
# 2) 환경 확인 + 자른 이미지 만들기 (OCR 없음)
import sys, os, json, time, random, csv
if WORK_DIR not in sys.path:
    sys.path.insert(0, WORK_DIR)
import importlib
import ocr_pipeline as op
import scripts.run_cosmetics_eval as ev
importlib.reload(ev)
from PIL import Image

missing = ev.weights_status()["missing"]
if missing:
    raise SystemExit("모델 파일이 없습니다: " + ", ".join(missing))
assert not any(900301 <= int(i) <= 900400 for i in IDS), "검증용 사진이 들어 있습니다"
files = {p.stem: p for p in Path(COS_DIR).iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"}}
lost = [i for i in IDS if i not in files]
if lost:
    raise SystemExit("사진이 없습니다: " + ", ".join(lost))
with open(COS_LABEL, encoding="utf-8-sig", newline="") as f:
    LABELS = {r["image_id"]: r for r in csv.DictReader(f)}
print("코드 버전:", (Path(WORK_DIR) / "CODE_VERSION.txt").read_text(encoding="utf-8").splitlines()[0])

OUT = Path(OUT_DIR); OUT.mkdir(parents=True, exist_ok=True)
PATHS = {}
for i in IDS:
    PATHS[(i, "full")] = files[i]
    rgb = op.decode_image(files[i])                 # 사진 방향(EXIF)을 바로잡은 뒤 자른다
    w, h = rgb.size
    for name, ratio in VARIANTS.items():
        if ratio is None or (name == "center40" and i not in IDS40):
            continue
        d = OUT / f"crops_{name}"; d.mkdir(exist_ok=True)
        path = d / f"{i}.png"
        if not path.exists():
            x0, y0 = round(w * (1 - ratio) / 2), round(h * (1 - ratio) / 2)
            rgb.crop((x0, y0, w - x0, h - y0)).save(path)   # PNG: 다시 압축하지 않음
        PATHS[(i, name)] = path
print(f"실행 대상: 원본 {len(IDS)} + 60% {len(IDS)} + 40% {len(IDS40)} = {len(PATHS)}건")

In [ ]:
# 3) 모델 준비 + 예열(기록 안 함): 첫 실행이 느린 것이 결과에 섞이지 않게
t0 = time.perf_counter()
ENGINE = op.initialize_engine(**ev.ENGINE_OPTIONS)
print(f"모델 준비 {time.perf_counter() - t0:.0f}초")
t0 = time.perf_counter()
op.predict_image_detailed(ENGINE, files[IDS[0]])
print(f"예열 1장 {time.perf_counter() - t0:.1f}초 -> 전체 예상 약 {(time.perf_counter() - t0) * len(PATHS) * 1.5 / 60:.0f}분")

In [ ]:
# 4) 본 실행: 끝난 것은 건너뛴다. 멈추면 이 셀만 다시 실행.
import ctypes
DUMP = OUT / "center_ocr.jsonl"
def done():
    keys = set()
    if DUMP.exists():
        data = DUMP.read_bytes()
        if data and not data.endswith(b"\n"):
            data = data[:data.rfind(b"\n") + 1]; DUMP.write_bytes(data)
        for line in data.decode("utf-8").splitlines():
            if line.strip():
                r = json.loads(line); keys.add((r["image_id"], r["variant"]))
    return keys
def keep_awake(on):
    try:
        ctypes.windll.kernel32.SetThreadExecutionState(0x80000000 | (0x00000001 if on else 0))
    except Exception:
        pass
rng = random.Random(SEED)
plan = []
for i in IDS:
    names = [n for n in VARIANTS if (i, n) in PATHS]
    rng.shuffle(names)                       # 사진마다 실행 순서를 섞는다
    plan += [(i, n) for n in names]
keys = done(); todo = [p for p in plan if p not in keys]
print(f"남은 {len(todo)}건 / 전체 {len(plan)}건", flush=True)
keep_awake(True); started = time.perf_counter()
try:
    for k, (i, name) in enumerate(todo, 1):
        path = PATHS[(i, name)]
        with Image.open(path) as im:
            size = im.size
        rec = ev.process_image(ENGINE, Path(path), MODE)
        ocr_official = sum(s["seconds"] for s in rec["stages"] if s["official"])
        out = {"image_id": i, "variant": name, "size": list(size), "label": LABELS[i]["final_date"],
               "prediction": rec["prediction"]["final_date"], "method": rec["method"], "attempts": rec["attempts"],
               "official_seconds": rec["official_seconds"], "ocr_official_seconds": ocr_official,
               "extra_seconds": rec["extra_seconds"], "stages": rec["stages"]}
        with DUMP.open("a", encoding="utf-8", newline="\n") as f:
            f.write(json.dumps(out, ensure_ascii=False) + "\n"); f.flush(); os.fsync(f.fileno())
        if k % 5 == 0 or k == len(todo):
            el = time.perf_counter() - started
            print(f"[{k}/{len(todo)}] {i} {name} 경과 {el/60:.0f}분, 남은 예상 {el/k*(len(todo)-k)/60:.0f}분", flush=True)
finally:
    keep_awake(False)
print("끝:", DUMP)

In [ ]:
# 5) 간단 요약 (OCR 없음). 자세한 분석은 Claude가 한다.
import statistics, collections
from scripts.evaluation_common import normalize_truth
rows = [json.loads(l) for l in DUMP.read_text(encoding="utf-8").splitlines() if l.strip()]
truth = {i: normalize_truth(LABELS[i])["final_date"] for i in IDS}
by = collections.defaultdict(dict)
for r in rows:
    by[r["variant"]][r["image_id"]] = r
for name in VARIANTS:
    R = by.get(name, {})
    if not R:
        continue
    ok = sum(R[i]["prediction"] == truth[i] for i in R)
    none = sum(R[i]["prediction"] == "NONE" for i in R)
    t = [R[i]["official_seconds"] for i in R]
    print(f"{name:9s} {len(R):2d}장  정답 {ok} ({100*ok/len(R):.0f}%)  날짜 못 찾음(NONE) {none}  "
          f"1장 시간 평균 {statistics.mean(t):.1f}초 / 중앙값 {statistics.median(t):.1f}초")
for name in ("center60", "center40"):
    R = by.get(name, {}); F = by.get("full", {})
    both = [i for i in R if i in F]
    if both:
        a = sum(F[i]["prediction"] == truth[i] and R[i]["prediction"] != truth[i] for i in both)
        b = sum(F[i]["prediction"] != truth[i] and R[i]["prediction"] == truth[i] for i in both)
        print(f"같은 사진 비교 (원본 vs {name}, {len(both)}장): 원본만 맞음 {a}장, {name}만 맞음 {b}장")
print("\nClaude에게 보낼 파일:", DUMP)